In [1]:
import re
from collections import Counter

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_PATTERN = re.compile(r"(?<!\w)@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
TOKEN_PATTERN = re.compile(
    r"URL|USER|[a-z]+(?:'[a-z]+)?|\d+|[:;=8][-^']?[)(/DPp]|[!?.,]+|[^\w\s]",
    re.IGNORECASE,
)

def normalize_tweet(text: str) -> str:
    """Normalize Twitter text while preserving useful sentiment signals."""
    text = text.lower()
    text = URL_PATTERN.sub(" URL ", text)
    text = MENTION_PATTERN.sub(" USER ", text)
    text = HASHTAG_PATTERN.sub(r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_tweet(text: str) -> list[str]:
    """Convert a raw tweet into tokens using the project's normalization rules."""
    normalized = normalize_tweet(text)
    return TOKEN_PATTERN.findall(normalized)

In [2]:
examples = [
    "I LOVE this phone!!! https://example.com",
    "@john this is #Amazing :)",
    "I don't like this at all 😡",
]

for tweet in examples:
    print("Original:", tweet)
    print("Tokens:  ", tokenize_tweet(tweet))
    print()

Original: I LOVE this phone!!! https://example.com
Tokens:   ['i', 'love', 'this', 'phone', '!!!', 'URL']

Original: @john this is #Amazing :)
Tokens:   ['USER', 'this', 'is', 'amazing', ':)']

Original: I don't like this at all 😡
Tokens:   ['i', "don't", 'like', 'this', 'at', 'all', '😡']



In [5]:
from datasets import ClassLabel

clean_data = clean_data.cast_column(
    "label",
    ClassLabel(names=["negative", "positive"])
)

first_split = clean_data.train_test_split(
    test_size=0.10,
    seed=42,
    stratify_by_column="label"
)

test_data = first_split["test"]

second_split = first_split["train"].train_test_split(
    test_size=1 / 9,
    seed=42,
    stratify_by_column="label"
)

train_data = second_split["train"]
val_data = second_split["test"]

print("Training rows:", len(train_data))
print("Validation rows:", len(val_data))
print("Test rows:", len(test_data))

c:\Users\lokes\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'clean_data' is not defined

In [ ]:
from collections import Counter

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

MIN_FREQ = 2
MAX_VOCAB_SIZE = 50_000

token_counts = Counter()

for tweet in train_data['text']:
    token_counts.update(tokenize_tweet(tweet))

print("Unique tokens before filtering:", len(token_counts))
print("Total token occurrences:", sum(token_counts.values()))

# Reserve two slots for special tokens
vocab_tokens = [
    token
    for token, count in token_counts.most_common()
    if count >= MIN_FREQ
][:MAX_VOCAB_SIZE - 2]

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
}

for token in vocab_tokens:
    if token not in vocab:
        vocab[token] = len(vocab)

print("Final vocabulary size:", len(vocab))
print("PAD ID:", vocab[PAD_TOKEN])
print("UNK ID:", vocab[UNK_TOKEN])

Unique tokens before filtering: 15
Total token occurrences: 18
Final vocabulary size: 4
PAD ID: 0
UNK ID: 1
